In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("../src")
from feature_engineering import build_features

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
df = build_features(filepath="../data/raw/synthetic_pricing_data.csv")
print(df.shape)
df.head(3)

In [ ]:
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls) == 0:
    print("No missing values — clean dataset ready for modelling")
else:
    print(nulls)

In [ ]:
sample = df[["demand", "demand_rolling_7d", "demand_rolling_30d",
             "demand_momentum"]].head(200)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

sample[["demand", "demand_rolling_7d", "demand_rolling_30d"]].plot(ax=axes[0])
axes[0].set_title("Demand: Raw vs Rolling Averages")
axes[0].set_ylabel("Demand")

sample["demand_momentum"].plot(ax=axes[1], color="purple")
axes[1].axhline(0, color="black", linestyle="--", linewidth=0.8)
axes[1].set_title("Demand Momentum (7d avg - 30d avg)")
axes[1].set_ylabel("Momentum")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["price_gap_pct"], kde=True, ax=axes[0], color="coral", bins=50)
axes[0].axvline(0, color="black", linestyle="--")
axes[0].set_title("Price Gap % (Our price vs Competitor)")
axes[0].set_xlabel("Gap % [+ve = we're more expensive]")

cheaper_pct = df["is_cheaper"].mean() * 100
axes[1].pie(
    [cheaper_pct, 100 - cheaper_pct],
    labels=["We're cheaper", "Competitor cheaper"],
    autopct="%1.1f%%",
    colors=["#4CAF50", "#FF7043"]
)
axes[1].set_title("Pricing Position vs Competitor")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df["price_elasticity"], kde=True, bins=60, color="steelblue")
plt.axvline(-1, color="red",   linestyle="--", label="Elastic threshold (-1)")
plt.axvline(0,  color="black", linestyle="--", label="Zero")
plt.title("Price Elasticity Distribution")
plt.xlabel("Elasticity  [< -1 = elastic, > -1 = inelastic]")
plt.legend()
plt.show()

print(f"Elastic products (< -1):   {(df['price_elasticity'] < -1).sum()}")
print(f"Inelastic products (> -1): {(df['price_elasticity'] > -1).sum()}")

In [ ]:
# Reconstruct stock_tier from one-hot columns for plotting
stock_cols = [c for c in df.columns if c.startswith("stock_tier_")]
if stock_cols:
    stock_counts = df[stock_cols].sum().sort_values(ascending=False)
    stock_counts.index = [c.replace("stock_tier_", "") for c in stock_counts.index]

    plt.figure(figsize=(7, 4))
    sns.barplot(x=stock_counts.index, y=stock_counts.values,
                palette=["#E53935", "#FB8C00", "#FDD835", "#43A047"])
    plt.title("Inventory Stock Tier Distribution")
    plt.ylabel("Count")
    plt.xlabel("Stock Tier")
    plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr_with_demand = numeric_df.corr()["demand"].drop("demand").sort_values()

plt.figure(figsize=(10, 8))
corr_with_demand.plot(kind="barh", color=[
    "#E53935" if v < 0 else "#43A047" for v in corr_with_demand.values
])
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Feature Correlation with Demand")
plt.xlabel("Pearson Correlation")
plt.tight_layout()
plt.show()

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/features.csv", index=False)
print(f"Saved: ../data/processed/features.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")